In [3]:
from langchain_openrouter import ChatOpenRouter
from langchain.chat_models import init_chat_model
from rich import print as rprint
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 获取大模型
# model = ChatOpenRouter(
#     model='~x-ai/grok-latest',
#     api_key=OPENROUTER_API_KEY,    # type: ignore
#     base_url=OPENROUTER_API_BASE,
#     max_tokens=1000
# )

model = init_chat_model(
    model='deepseek-flash',
    extra_body={"thinking": {"type": "disabled"}},
)

/Users/refone/Coding/ai/langchain-learn/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [4]:
from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名30岁的软件工程师")

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [5]:
from typing import Literal, Optional


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["紧急","一般"] = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency='紧急'
